# Free CogVideoX Image-to-Video Backend
Run every cell from top to bottom. Keep this Colab tab open while generating videos from the website.

This uses the open-source `THUDM/CogVideoX-5b-I2V` model on a free Colab GPU. Free Colab sessions are temporary and generations can be slow.

In [ ]:
!nvidia-smi
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg fastapi uvicorn python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, uuid, threading, subprocess, re, time
from pathlib import Path
import torch
from PIL import Image
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video

MODEL_ID = 'THUDM/CogVideoX-5b-I2V'
print('Loading CogVideoX. This can take several minutes the first time...')
pipe = CogVideoXImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print('MODEL READY ✅')

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False, allow_methods=['*'], allow_headers=['*'])
jobs = {}
work = Path('/content/video_jobs')
work.mkdir(exist_ok=True)

def run_job(job_id, image_path, prompt, style):
    try:
        jobs[job_id] = {'status':'running','message':'CogVideoX is animating your character on the free GPU...'}
        image = Image.open(image_path).convert('RGB')
        full_prompt = f'{style} character music video. Preserve the same character identity, face, colors, markings and proportions. {prompt}. Smooth expressive motion, playful performance, cinematic movement, consistent character.'
        frames = pipe(
            prompt=full_prompt,
            image=image,
            height=480,
            width=720,
            num_frames=49,
            num_inference_steps=12,
            guidance_scale=6.0,
            use_dynamic_cfg=True,
            generator=torch.Generator().manual_seed(42)
        ).frames[0]
        output = work / f'{job_id}.mp4'
        export_to_video(frames, str(output), fps=8)
        jobs[job_id] = {'status':'completed','message':'Video ready','video_url':f'/video/{job_id}'}
    except Exception as e:
        jobs[job_id] = {'status':'failed','error':str(e)}

@app.get('/')
def root():
    return {'ok':True,'model':MODEL_ID,'message':'Free CogVideoX backend is ready'}

@app.post('/submit')
async def submit(image: UploadFile = File(...), prompt: str = Form('Make the character dance playfully for the camera'), style: str = Form('Funny')):
    job_id = uuid.uuid4().hex
    image_path = work / f'{job_id}_{image.filename or "image.png"}'
    image_path.write_bytes(await image.read())
    jobs[job_id] = {'status':'queued','message':'Queued on free Colab GPU...'}
    threading.Thread(target=run_job, args=(job_id, str(image_path), prompt, style), daemon=True).start()
    return {'ok':True,'job_id':job_id}

@app.get('/status/{job_id}')
def status(job_id: str):
    if job_id not in jobs:
        return {'ok':False,'error':'Unknown job'}
    return {'ok':True, **jobs[job_id]}

@app.get('/video/{job_id}')
def video(job_id: str):
    output = work / f'{job_id}.mp4'
    if not output.exists():
        return {'ok':False,'error':'Video not ready'}
    return FileResponse(str(output), media_type='video/mp4', filename=f'{job_id}.mp4')

server = threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True)
server.start()
time.sleep(2)

proc = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in proc.stdout:
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print('\n\n============================================')
        print('COPY THIS URL INTO YOUR WEBSITE:')
        print(public_url)
        print('============================================\n')
        break

if not public_url:
    print('Tunnel URL was not detected. Re-run this cell.')

## Keep Colab open
Copy the `https://xxxxx.trycloudflare.com` URL shown above into **Free AI backend** on your website.

If Colab disconnects, run the last cell again and paste the new URL into the website.